# Day 4 — Agents in action

A complete mini-system, end to end: **support-ticket triage** — classify with
an agent + `response_format`, route with rules, draft with a model, and then
**prove it works with an eval**.

This is the smallest honest version of the production systems in today's deck.

*GIU AI Connects · Amr Saleh · iHQ Tech*

## 0 · Setup

One `uv` environment for the whole week (see `README.md`): `uv sync`, then run
Jupyter with `uv run jupyter lab`.

Put your API key in a `.env` file next to this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
```

> Using another provider? Swap the model string below — e.g. `"openai:gpt-4.1"`
> with `OPENAI_API_KEY` in `.env`. Everything else stays identical: that is the
> point of the harness.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

MODEL = "anthropic:claude-haiku-4-5"   # swap provider here if needed
llm = init_chat_model(MODEL)

llm.invoke("Say 'ready' and nothing else.").content

## 1 · The case: IT-helpdesk triage

Design first: a **schema** for what comes out of classification; **rules** as
code (security ALWAYS goes to a human); the model drafts only the safe class.

Day 2's differentiation, applied: classification is **one typed step inside a
workflow we control** — no tools, no planning. So it's the **direct form**,
not an agent.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Triage(BaseModel):
    """Classification of one IT support ticket."""
    category: Literal["password", "vpn", "software", "hardware", "security", "other"]
    confidence: int = Field(ge=0, le=100, description="How sure you are, 0-100")
    summary: str = Field(description="One line: what the user needs")

classifier = llm.with_structured_output(Triage)   # the direct form

def classify(ticket, extra_instructions=""):
    return classifier.invoke(
        f"Classify this IT support ticket. {extra_instructions}\n\nTicket: {ticket}")

TICKETS = [
    "Locked out of my account again, password reset link never arrives.",
    "VPN drops every 20 minutes when I'm on the dorm wifi.",
    "Need a Tableau licence for my thesis project.",
    "Suspicious login alert from a device in another country?!",
    "My laptop screen flickers when I open the lid.",
    "Everything is slow since Monday. Everything.",
    "Someone sent a link to 'verify my student account' — I clicked it.",
    "How do I connect the beamer in room C01 to HDMI?",
]

In [ ]:
# routing is CODE — the threshold and the security override are decisions you sign
AUTO_THRESHOLD = 85

def route(t: Triage) -> str:
    if t.category == "security":
        return "HUMAN (security desk)"      # always — no threshold applies
    if t.confidence >= AUTO_THRESHOLD and t.category in ("password", "vpn", "software"):
        return "auto-reply"
    return "HUMAN (helpdesk)"

def draft_reply(ticket: str, t: Triage) -> str:
    out = llm.invoke(
        f"Draft a 2-sentence friendly IT reply for this {t.category} issue. "
        f"Give the single most likely fix. Ticket: {ticket}")
    return out.content

results = []
for ticket in TICKETS:
    t = classify(ticket)
    r = route(t)
    results.append((ticket, t, r))
    print(f"{t.category:9} {t.confidence:3}% → {r:22} | {t.summary[:50]}")

**Discuss** (from the deck): a security incident misfiled as a password reset
is not an accuracy problem — it's a *cost asymmetry* problem. That is why the
security override is unconditional code, not a confidence score.

## 2 · Evals — prove it, don't feel it

Five minutes of building an eval beats five weeks of "seems fine". Cases =
real past inputs + what *right* looks like.

In [ ]:
EVAL_CASES = [
    ("Locked out of my account again, password reset link never arrives.", "password"),
    ("VPN drops every 20 minutes when I'm on the dorm wifi.",              "vpn"),
    ("Suspicious login alert from a device in another country?!",          "security"),
    ("Someone sent a link to 'verify my student account' — I clicked it.", "security"),
    ("Need a Tableau licence for my thesis project.",                      "software"),
    ("My laptop screen flickers when I open the lid.",                     "hardware"),
]

def run_eval(extra_instructions=""):
    passed, failures = 0, []
    for ticket, expected in EVAL_CASES:
        t = classify(ticket, extra_instructions)
        ok = t.category == expected
        # the REAL check: security must never route to auto-reply
        if expected == "security" and route(t) == "auto-reply":
            ok = False
        passed += ok
        if not ok:
            failures.append((ticket[:45], expected, t.category, route(t)))
    print(f"pass rate: {passed}/{len(EVAL_CASES)}")
    for f in failures:
        print("  FAIL:", f)
    return failures

failures = run_eval()

In [ ]:
# found a failure? fix the PROMPT, not the labels — then run it again
run_eval("Anything involving suspicious logins, phishing links, or clicked "
         "links is category 'security', regardless of what else it mentions.")

That loop — run, read failures, fix the prompt, run again — is the whole
discipline. The eval table is your **acceptance test**.

**Your turn** — add two adversarial cases the current prompt gets wrong (mixed
issues are a good hunting ground: "VPN broken AND weird login alert"). Fix the
prompt until green — without breaking the other cases.

In [ ]:
# your adversarial cases here


**Optional — LLM as judge:** hard checks can't score a *draft reply's* quality.
A second structured call can.

In [ ]:
class Verdict(BaseModel):
    """Judgement of one drafted reply."""
    helpful: bool
    safe: bool = Field(description="No credentials requested, no links, no promises IT can't keep")
    reason: str

judge = llm.with_structured_output(Verdict)   # direct form: no tools, no loop

ticket, t, _ = results[0]
draft = draft_reply(ticket, t)
print(draft, "\n")
judge.invoke(f"Judge this IT support draft.\nTicket: {ticket}\nDraft: {draft}")

## 3 · Take-home — turn one task into an agent

Answer on one page (deck has the full worksheet):

1. **The task** — something you repeat weekly
2. **Inputs** — what you hand over to start
3. **Steps** — explain it to someone new
4. **Tools** — what must it look up or touch?
5. **Schema** — what structured thing comes out?
6. **Guardrails** — what must never happen alone?
7. **Eval cases** — five real past examples + what "right" was
8. **Workflow or agent?** — least powerful abstraction. Argue it.

Then build the skeleton with day-2 pieces and **bring it tomorrow with a
trace** — office hours exist for the broken half.

---
*Friday: office hours. Bring your bugs, your traces, your ideas.*